# BYOB seed data preparation

This notebook runs **seed preparation** for the Nemotron BYOB pipeline: it pairs few-shot examples from a Hugging Face MCQ benchmark (e.g. MMLU) with domain text under ``input_dir/<target_subject>/*.txt`` and writes ``{output_dir}/{expt_name}/seed.parquet``.

Corpus text comes from **Wikipedia** via ``download_wikipedia_category.py`` in this folder. This example uses [Category:Finance in India](https://en.wikipedia.org/wiki/Category:Finance_in_India) with **depth 2** (root category + two levels of subcategories) and **at most 30 articles**.

**Requirements:** ``omegaconf``, ``pydantic``, ``numpy``, ``pandas``, ``pyarrow``, ``datasets``, ``requests``; set ``PYTHONPATH`` to the Nemotron repo ``src`` directory. You need network access to Hugging Face and Wikipedia.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

def _find_nemotron_repo() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "src" / "nemotron").is_dir():
            return p
    raise FileNotFoundError(
        "Could not find Nemotron repo (folder with src/nemotron). "
        "Open this notebook from the repo or set cwd to use-case-examples/Bring-your-own-benchmark."
    )

REPO = _find_nemotron_repo()
HERE = (REPO / "use-case-examples" / "Bring-your-own-benchmark").resolve()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.environ.setdefault("PYTHONPATH", str(SRC))
print("REPO:", REPO)
print("HERE (this example):", HERE)
print("PYTHONPATH:", os.environ.get("PYTHONPATH"))

## 1) Download Wikipedia text (BYOB corpus layout)

The script writes ``<output-dir>/<target-subject>/*.txt`` for pages discovered under the category tree. Here: **Finance in India**, **``--depth 2``** (not ``--recursive``; depth and recursive are different modes in the script), **``--max-articles 30``**. Set **``input_dir``** in the BYOB config to the **``--output-dir``** value and match **``--target-subject``** in **``target_source_mapping``**.

In [ ]:
from subprocess import run

DOWNLOAD_SCRIPT = HERE / "download_wikipedia_category.py"
WIKI_ROOT = HERE / "input" / "wikipedia_finance_in_india"
TARGET_KEY = "finance_india"  # must match --target-subject (see download_wikipedia_category.py --help)

WIKI_ROOT.parent.mkdir(parents=True, exist_ok=True)

completed = run(
    [
        sys.executable,
        str(DOWNLOAD_SCRIPT),
        "--output-dir",
        str(WIKI_ROOT),
        "--target-subject",
        TARGET_KEY,
        "--category",
        "Finance in India",
        "--max-articles",
        "30",
        "--depth",
        "2",
        "--delay",
        "0.35",
    ],
    check=True,
    cwd=str(HERE),
)
corpus_root = WIKI_ROOT  # input_dir = parent of <TARGET_KEY>/*.txt
print("Corpus (input_dir for BYOB):", corpus_root)
txts = list((corpus_root / TARGET_KEY).glob("*.txt"))
print(f"Wrote {len(txts)} article .txt under {corpus_root / TARGET_KEY}")
print("Sample:", [p.name for p in txts[:8]])

## 2) Build OmegaConf and run ``prepare_byob_seed``

``source_subjects`` uses a single MMLU subject for a smaller HF download. Use ``[]`` for all subjects in full runs (slow).

In [ ]:
from omegaconf import OmegaConf
from nemotron.customization_recipes.data_prep import prepare_byob_seed

out = HERE / "output" / "byob_seed_notebook"
out.mkdir(parents=True, exist_ok=True)

cfg = OmegaConf.create(
    {
        "expt_name": "notebook_byob_seed",
        "input_dir": str(corpus_root),
        "output_dir": str(out),
        "hf_dataset": "cais/mmlu",
        "subset": "all",
        "split": "test",
        "source_subjects": ["anatomy"],
        "target_source_mapping": {TARGET_KEY: {"subjects": ["anatomy"]}},
        "few_shot_samples_per_query": 1,
        "queries_per_target_subject_document": 1,
        "num_questions_per_query": 2,
        "random_seed": 42,
        "chunking_config": {"window_size": None},
    }
)

result = prepare_byob_seed(cfg)
result

In [ ]:
import pandas as pd

p = Path(result["seed_path"])
assert p.is_file(), p
df = pd.read_parquet(p)
df.head(), df.shape